# Tennessee Eastman Process (TEP) — Fault Dataset Exploration

**Dataset**: 52 process variables (41 measured + 11 manipulated), 21 fault types, sampled every 3 minutes.  
**Files**: FaultFree Training/Testing + Faulty Training/Testing (read directly from zip).

In [ ]:
import zipfile
import io
import pyreadr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams["figure.dpi"] = 120

ZIP_PATH = "../datasets/TEP/Harvard/dataverse_files.zip"

XMEAS = [f"xmeas_{i}" for i in range(1, 42)]
XMV   = [f"xmv_{i}"   for i in range(1, 12)]
FEATURES = XMEAS + XMV

def load_rdata(zip_path, filename):
    with zipfile.ZipFile(zip_path) as z:
        with z.open(filename) as f:
            data = f.read()
    result = pyreadr.read_r(io.BytesIO(data))
    return list(result.values())[0]

print("Libraries loaded.")

## 1. Load Data

In [ ]:
print("Loading FaultFree Training...")
ff_train = load_rdata(ZIP_PATH, "TEP_FaultFree_Training.RData")

print("Loading FaultFree Testing...")
ff_test  = load_rdata(ZIP_PATH, "TEP_FaultFree_Testing.RData")

print("Loading Faulty Training...")
f_train  = load_rdata(ZIP_PATH, "TEP_Faulty_Training.RData")

print("Loading Faulty Testing...")
f_test   = load_rdata(ZIP_PATH, "TEP_Faulty_Testing.RData")

print("\nDone.")
for name, df in [("FF Train", ff_train), ("FF Test", ff_test),
                 ("Faulty Train", f_train), ("Faulty Test", f_test)]:
    print(f"  {name:15s}: {df.shape[0]:>8,} rows × {df.shape[1]} cols")

## 2. Basic Exploration

In [ ]:
# Schema and data types
print("Columns:", ff_train.columns.tolist())
print("\nData types:")
print(ff_train.dtypes)

In [ ]:
# Missing values
print("Missing values (fault-free training):")
nulls = ff_train.isnull().sum()
print(nulls[nulls > 0] if nulls.any() else "  None")

In [ ]:
# Descriptive statistics for fault-free training features
ff_train[FEATURES].describe().T.round(3)

In [ ]:
# Fault type distribution in faulty training data
fault_counts = f_train.groupby("faultNumber")["simulationRun"].nunique().reset_index()
fault_counts.columns = ["faultNumber", "runs"]

fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(fault_counts["faultNumber"].astype(int), fault_counts["runs"], color="steelblue")
ax.set_xlabel("Fault Number (IDV)")
ax.set_ylabel("Simulation Runs")
ax.set_title("Number of Simulation Runs per Fault Type (Faulty Training)")
ax.set_xticks(fault_counts["faultNumber"].astype(int))
plt.tight_layout()
plt.show()

## 3. Time Series — Normal Operation

One simulation run from the fault-free training set, showing all 52 process variables.

In [ ]:
# Pick one run and plot all features
run_id = 1
run = ff_train[ff_train["simulationRun"] == run_id].sort_values("sample")
time_h = run["sample"] * 3 / 60  # convert samples to hours

n_xmeas, n_xmv = len(XMEAS), len(XMV)
fig, axes = plt.subplots(n_xmeas, 1, figsize=(14, n_xmeas * 0.9), sharex=True)
fig.suptitle(f"Fault-Free Training — Run {run_id} — xmeas_1 to xmeas_41", y=1.001, fontsize=11)
for ax, col in zip(axes, XMEAS):
    ax.plot(time_h, run[col], lw=0.7, color="royalblue")
    ax.set_ylabel(col, fontsize=6, rotation=0, labelpad=40, va="center")
    ax.tick_params(axis="y", labelsize=6)
axes[-1].set_xlabel("Time (hours)")
plt.tight_layout()
plt.show()

In [ ]:
# Manipulated variables (xmv_1–11)
fig, axes = plt.subplots(n_xmv, 1, figsize=(14, n_xmv * 1.1), sharex=True)
fig.suptitle(f"Fault-Free Training — Run {run_id} — xmv_1 to xmv_11", fontsize=11)
for ax, col in zip(axes, XMV):
    ax.plot(time_h, run[col], lw=0.8, color="darkorange")
    ax.set_ylabel(col, fontsize=7, rotation=0, labelpad=40, va="center")
    ax.tick_params(axis="y", labelsize=7)
axes[-1].set_xlabel("Time (hours)")
plt.tight_layout()
plt.show()

## 4. Fault vs. Normal — Time Series Comparison

Compare the same 6 key variables across fault-free and a few fault types (IDV 1, 4, 5).  
Faults are injected at sample 160 (≈ 8 hours).

In [ ]:
COMPARE_VARS  = ["xmeas_1", "xmeas_7", "xmeas_9", "xmeas_11", "xmv_3", "xmv_4"]
FAULT_NUMBERS = [1, 4, 5]
FAULT_INJECTION_SAMPLE = 160

# Use run 1 from each group (faulty training run IDs are per fault type)
ff_run  = ff_test[ff_test["simulationRun"] == 1].sort_values("sample")
faulty_runs = {
    fnum: f_test[(f_test["faultNumber"] == fnum) & (f_test["simulationRun"] == 1)].sort_values("sample")
    for fnum in FAULT_NUMBERS
}

fig, axes = plt.subplots(len(COMPARE_VARS), 1, figsize=(14, len(COMPARE_VARS) * 2.2), sharex=True)
colors = {"normal": "royalblue", 1: "tomato", 4: "seagreen", 5: "darkorchid"}

for ax, var in zip(axes, COMPARE_VARS):
    t = ff_run["sample"] * 3 / 60
    ax.plot(t, ff_run[var], lw=1, label="Normal", color=colors["normal"], alpha=0.85)
    for fnum, frun in faulty_runs.items():
        tf = frun["sample"] * 3 / 60
        ax.plot(tf, frun[var], lw=1, label=f"IDV {fnum}", color=colors[fnum], alpha=0.85)
    ax.axvline(FAULT_INJECTION_SAMPLE * 3 / 60, color="black", lw=1, ls="--", alpha=0.5)
    ax.set_ylabel(var, fontsize=9)
    ax.legend(fontsize=7, loc="upper right", ncol=4)

axes[-1].set_xlabel("Time (hours)")
fig.suptitle("Fault-Free vs. IDV 1/4/5 — Testing Set (Run 1)", fontsize=12)
plt.tight_layout()
plt.show()

## 5. Feature Distributions — Normal vs. All Faults

Box plots and violin plots comparing the distribution of each feature group between normal and faulty operation (post-fault injection window only).

In [ ]:
# Build a combined dataframe: normal vs. faulty (post-injection only)
ff_sample = ff_train[FEATURES].copy()
ff_sample["label"] = "Normal"

faulty_post = f_train[f_train["sample"] > FAULT_INJECTION_SAMPLE][FEATURES].copy()
faulty_post["label"] = "Faulty"

# Downsample for plotting speed
rng = np.random.default_rng(42)
n = 5000
idx_ff = rng.choice(len(ff_sample), size=min(n, len(ff_sample)), replace=False)
idx_f  = rng.choice(len(faulty_post), size=min(n, len(faulty_post)), replace=False)
combined = pd.concat([ff_sample.iloc[idx_ff], faulty_post.iloc[idx_f]], ignore_index=True)

# Box plots — xmeas group (first 22 continuous measurements)
plot_vars = XMEAS[:22]
fig, axes = plt.subplots(4, 6, figsize=(18, 12))
axes = axes.flatten()
for ax, var in zip(axes, plot_vars):
    sns.boxplot(data=combined, x="label", y=var, ax=ax,
                palette={"Normal": "royalblue", "Faulty": "tomato"}, width=0.5, linewidth=0.8)
    ax.set_title(var, fontsize=8)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="x", labelsize=7)
for ax in axes[len(plot_vars):]:
    ax.set_visible(False)
fig.suptitle("Distribution: Normal vs. Faulty — xmeas_1 to xmeas_22", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Violin plots — manipulated variables (xmv_1–11)
fig, axes = plt.subplots(2, 6, figsize=(18, 7))
axes = axes.flatten()
for ax, var in zip(axes, XMV):
    sns.violinplot(data=combined, x="label", y=var, ax=ax,
                   palette={"Normal": "royalblue", "Faulty": "tomato"}, linewidth=0.8)
    ax.set_title(var, fontsize=8)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="x", labelsize=7)
for ax in axes[len(XMV):]:
    ax.set_visible(False)
fig.suptitle("Distribution: Normal vs. Faulty — xmv_1 to xmv_11 (Manipulated Variables)", fontsize=12)
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap — Normal Operation

In [ ]:
corr = ff_train[FEATURES].sample(10000, random_state=42).corr()

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, ax=ax,
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.2, annot=False, square=True,
    cbar_kws={"shrink": 0.7, "label": "Pearson r"},
)
ax.set_title("Feature Correlation — Fault-Free Training (10k sample)", fontsize=12)
plt.tight_layout()
plt.show()

## 7. Per-Fault Mean Deviation from Normal

For each fault type, compute the mean absolute deviation of each feature from the fault-free mean.  
This shows which variables are most discriminative for each fault.

In [ ]:
# Normalize features to unit variance using fault-free training stats
ff_mean = ff_train[FEATURES].mean()
ff_std  = ff_train[FEATURES].std().replace(0, 1)

# Post-injection faulty data, normalized
faulty_post_norm = (
    f_train[f_train["sample"] > FAULT_INJECTION_SAMPLE][["faultNumber"] + FEATURES]
    .copy()
)
faulty_post_norm[FEATURES] = (faulty_post_norm[FEATURES] - ff_mean) / ff_std

# Mean absolute normalized deviation per fault type
dev = (
    faulty_post_norm.groupby("faultNumber")[FEATURES]
    .apply(lambda g: g.abs().mean())
)

fig, ax = plt.subplots(figsize=(18, 8))
sns.heatmap(
    dev.T, ax=ax,
    cmap="YlOrRd", linewidths=0.3,
    cbar_kws={"label": "Mean |z-score| (post-injection)"},
    yticklabels=True,
)
ax.set_xlabel("Fault Number (IDV)")
ax.set_ylabel("Feature")
ax.set_title("Feature Sensitivity per Fault Type (normalized deviation from normal mean)")
ax.tick_params(axis="y", labelsize=6)
plt.tight_layout()
plt.show()

## 8. Average Fault Trajectory

Show how the process mean evolves over time for normal vs. selected faults, averaged across all runs.  
The dashed vertical line marks the fault injection point.

In [ ]:
TRAJ_VARS   = ["xmeas_1", "xmeas_4", "xmeas_7", "xmeas_11"]
TRAJ_FAULTS = [1, 2, 5, 11]

# Mean trajectory per sample across all runs
ff_traj = ff_test.groupby("sample")[TRAJ_VARS].mean()

fig, axes = plt.subplots(len(TRAJ_VARS), 1, figsize=(13, len(TRAJ_VARS) * 2.5), sharex=True)
palette = plt.cm.tab10.colors

for ax, var in zip(axes, TRAJ_VARS):
    t = ff_traj.index * 3 / 60
    ax.plot(t, ff_traj[var], lw=1.5, color="royalblue", label="Normal", zorder=3)
    for i, fnum in enumerate(TRAJ_FAULTS):
        f_traj = f_test[f_test["faultNumber"] == fnum].groupby("sample")[var].mean()
        tf = f_traj.index * 3 / 60
        ax.plot(tf, f_traj, lw=1, color=palette[i + 1], label=f"IDV {fnum}", alpha=0.85)
    ax.axvline(FAULT_INJECTION_SAMPLE * 3 / 60, color="black", lw=1, ls="--", alpha=0.4)
    ax.set_ylabel(var, fontsize=9)
    ax.legend(fontsize=7, loc="upper right", ncol=5)

axes[-1].set_xlabel("Time (hours)")
fig.suptitle("Mean Trajectory — Normal vs. Selected Faults (Testing Set)", fontsize=12)
plt.tight_layout()
plt.show()